# Quick Sort — Simulación interactiva con Números

En esta simulacion se modeló y se hizo un modelo interactivo con el que se puede editar el arreglo y cantidad de números, en este modelo se usa la partición de Lomuto para el pivote (ocupando el povite como el último elemento del subarreglo) por la facilidad visual.

**Cómo usarlo**
1. Ejecuta la primera celda (define el simulador).
2. Ejecuta la segunda (`simulador_quick_sort_numeros()`) y aparece la interfaz.
3. Elige la cantidad de números y pulsa **Generar Nuevo**, o escribe tus propios números y pulsa **Usar mis números**.

**Cómo se lee la animación:** el **pivote** es el último elemento del subarreglo (naranja). Se recorre el subarreglo comparando cada elemento (rojo) con el pivote; los menores se van juntando a la izquierda (azul). Al final el pivote se coloca en su posición definitiva (verde) y se repite el proceso a cada lado.

In [1]:
import uuid
from IPython.display import HTML, display

# Todo el simulador (interfaz + algoritmo) vive en este bloque de HTML/CSS/JavaScript.
# Funciona en Jupyter, JupyterLab, Colab y VS Code.
PLANTILLA = r"""
<style>
#__UID__ { font-family: Segoe UI, Roboto, Arial, sans-serif; border: 1px solid #ddd;
  border-radius: 10px; padding: 14px 16px; max-width: 980px; background: #fff; color: #222; }
#__UID__ .titulo { font-size: 18px; margin-bottom: 10px; }
#__UID__ .fila { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; margin: 6px 0; }
#__UID__ input[type=number] { width: 60px; padding: 3px 4px; }
#__UID__ input[type=text] { width: 260px; padding: 3px 6px; }
#__UID__ button { padding: 5px 12px; border: 1px solid #555; border-radius: 3px;
  background: #f2f2f2; color: #222; cursor: pointer; font-size: 13px; }
#__UID__ button:hover:not(:disabled) { background: #e2e2e2; }
#__UID__ button:disabled { opacity: .45; cursor: default; }
#__UID__ .panel { position: relative; height: 190px; background: #1c1c1c; border-radius: 6px;
  margin: 10px 0; overflow: hidden; }
#__UID__ .casilla { position: absolute; top: 46px; height: 44px; line-height: 40px; text-align: center;
  box-sizing: border-box; border: 2px solid #4a4a4a; border-radius: 4px; background: #2e2e2e;
  color: #fff; font-weight: 700; font-size: 16px; transition: left .45s ease, opacity .3s, background .2s; }
#__UID__ .casilla.rng { border-color: #9a9a9a; }
#__UID__ .casilla.out { opacity: .3; }
#__UID__ .casilla.low { background: #0d47a1; border-color: #42a5f5; }
#__UID__ .casilla.cmp { background: #8b0000; border-color: #ff5252; }
#__UID__ .casilla.pv  { background: #b45309; border-color: #ffb74d; }
#__UID__ .casilla.ok  { background: #1b5e20; border-color: #66bb6a; }
#__UID__ .marca { position: absolute; top: 96px; width: 40px; text-align: center; font-size: 12px;
  font-weight: 700; color: #ffb74d; transition: left .45s ease; }
#__UID__ .texto { position: absolute; left: 12px; right: 12px; bottom: 12px; text-align: center;
  color: #e0e0e0; font-size: 14px; min-height: 36px; }
#__UID__ .contador { margin-left: auto; color: #555; font-size: 14px; }
#__UID__ .leyenda { display: flex; gap: 14px; flex-wrap: wrap; font-size: 12px; color: #444; margin-top: 8px; }
#__UID__ .leyenda i { display: inline-block; width: 12px; height: 12px; border-radius: 2px;
  margin-right: 5px; vertical-align: -2px; border: 2px solid; box-sizing: border-box; }
</style>

<div id="__UID__">
  <div class="titulo"> <b>Quick Sort</b> — Simulación con Números</div>
  <div class="fila">
    <span>Cantidad de números (5 a 20):</span>
    <input type="number" class="cant" min="5" max="20" value="12">
    <button class="nuevo">↻ Generar Nuevo</button>
  </div>
  <div class="fila">
    <span>O escribe los tuyos:</span>
    <input type="text" class="mios" placeholder="ej: 7, 2, 9, 4, 1, 8">
    <button class="usar">Usar mis números</button>
  </div>
  <div class="panel">
    <div class="texto"></div>
  </div>
  <div class="fila">
    <button class="ant">⏮ Anterior</button>
    <button class="play">▶ Reproducir</button>
    <button class="sig">Siguiente ⏭</button>
    <span class="contador">Paso: 1 / 1</span>
  </div>
  <div class="leyenda">
    <span><i style="background:#b45309;border-color:#ffb74d"></i>pivote</span>
    <span><i style="background:#8b0000;border-color:#ff5252"></i>comparando / intercambiando</span>
    <span><i style="background:#0d47a1;border-color:#42a5f5"></i>menores que el pivote</span>
    <span><i style="background:#2e2e2e;border-color:#9a9a9a"></i>subarreglo actual</span>
    <span><i style="background:#1b5e20;border-color:#66bb6a"></i>ya en su lugar</span>
  </div>
</div>

<script>
(function () {
  var raiz = document.getElementById("__UID__");
  var panel = raiz.querySelector(".panel");
  var textoEl = raiz.querySelector(".texto");
  var contador = raiz.querySelector(".contador");
  var btnAnt = raiz.querySelector(".ant");
  var btnSig = raiz.querySelector(".sig");
  var btnPlay = raiz.querySelector(".play");
  var inCant = raiz.querySelector(".cant");
  var inMios = raiz.querySelector(".mios");

  var pasos = [], casillas = [], marca = null, actual = 0, timer = null;

  // <qs>
  // Quick sort (partición de Lomuto, pivote = último elemento). Cada paso es una "foto".
  function quicksortPasos(valores) {
    var a = valores.map(function (v, id) { return { id: id, v: v }; });
    var pasos = [], ordenados = {};
    function nz(x) { return x === undefined ? null : x; }
    function foto(txt, lo, hi, pv, i, j, sw) {
      pasos.push({
        orden: a.map(function (x) { return x.id; }),
        lo: nz(lo), hi: nz(hi), pv: nz(pv), i: nz(i), j: nz(j), sw: !!sw,
        ord: Object.assign({}, ordenados), txt: txt
      });
    }
    function qs(lo, hi) {
      if (lo > hi) return;
      if (lo === hi) {
        ordenados[lo] = true;
        foto("Un solo elemento: " + a[lo].v + " ya está en su lugar.", lo, hi);
        return;
      }
      var pivote = a[hi].v;
      foto("Subarreglo [" + lo + ".." + hi + "]. Elijo como pivote el último elemento: " + pivote + ".", lo, hi, hi);
      var i = lo - 1;
      for (var j = lo; j < hi; j++) {
        foto("Comparo " + a[j].v + " con el pivote " + pivote + ".", lo, hi, hi, i, j);
        if (a[j].v < pivote) {
          i++;
          if (i !== j) {
            var t = a[i]; a[i] = a[j]; a[j] = t;
            foto(a[i].v + " < " + pivote + ": lo intercambio hacia la zona de menores (posición " + i + ").", lo, hi, hi, i, j, true);
          } else {
            foto(a[i].v + " < " + pivote + ": ya está en la zona de menores.", lo, hi, hi, i, j);
          }
        } else {
          foto(a[j].v + " ≥ " + pivote + ": se queda a la derecha.", lo, hi, hi, i, j);
        }
      }
      var t2 = a[i + 1]; a[i + 1] = a[hi]; a[hi] = t2;
      var p = i + 1;
      ordenados[p] = true;
      foto("Coloco el pivote " + pivote + " en su posición final: " + p + ".", lo, hi, p);
      qs(lo, p - 1);
      qs(p + 1, hi);
    }
    foto("Arreglo inicial.");
    qs(0, a.length - 1);
    foto("¡Arreglo ordenado!");
    return pasos;
  }
  // </qs>

  function ancho() { return panel.clientWidth || 800; }

  function crearCasillas(valores) {
    casillas.forEach(function (c) { panel.removeChild(c); });
    if (marca) { panel.removeChild(marca); }
    casillas = valores.map(function (v) {
      var d = document.createElement("div");
      d.className = "casilla";
      d.textContent = v;
      panel.appendChild(d);
      return d;
    });
    marca = document.createElement("div");
    marca.className = "marca";
    marca.textContent = "▲ pivote";
    panel.appendChild(marca);
  }

  function mostrar(k) {
    actual = k;
    var p = pasos[k], n = casillas.length;
    var slot = ancho() / n;
    var w = Math.max(24, Math.min(46, slot - 6));
    p.orden.forEach(function (id, pos) {
      var c = casillas[id];
      c.style.width = w + "px";
      c.style.left = (pos * slot + (slot - w) / 2) + "px";
      var cls = "casilla";
      if (p.ord[pos]) cls += " ok";
      else if (pos === p.pv) cls += " pv";
      else if (pos === p.j || (p.sw && pos === p.i)) cls += " cmp";
      else if (p.i !== null && p.lo !== null && pos >= p.lo && pos <= p.i) cls += " low";
      else if (p.lo !== null && pos >= p.lo && pos <= p.hi) cls += " rng";
      else if (p.lo !== null) cls += " out";
      c.className = cls;
    });
    if (p.pv !== null && !p.ord[p.pv]) {
      marca.style.display = "block";
      marca.style.width = w + "px";
      marca.style.left = (p.pv * slot + (slot - w) / 2) + "px";
    } else {
      marca.style.display = "none";
    }
    textoEl.textContent = p.txt;
    contador.textContent = "Paso: " + (k + 1) + " / " + pasos.length;
    btnAnt.disabled = (k === 0);
    btnSig.disabled = (k === pasos.length - 1);
  }

  function pausar() {
    if (timer) { clearInterval(timer); timer = null; }
    btnPlay.textContent = "▶ Reproducir";
  }

  function reproducir() {
    if (actual >= pasos.length - 1) mostrar(0);
    btnPlay.textContent = "⏸ Pausar";
    timer = setInterval(function () {
      if (actual >= pasos.length - 1) { pausar(); return; }
      mostrar(actual + 1);
    }, 700);
  }

  function cargar(valores) {
    pausar();
    crearCasillas(valores);
    pasos = quicksortPasos(valores);
    mostrar(0);
  }

  function aleatorios() {
    var n = parseInt(inCant.value, 10);
    if (isNaN(n)) n = 12;
    n = Math.max(5, Math.min(20, n));
    inCant.value = n;
    var v = [];
    for (var i = 0; i < n; i++) v.push(Math.floor(Math.random() * 99) + 1);
    cargar(v);
  }

  function propios() {
    var v = inMios.value.split(/[\s,;]+/).filter(Boolean).map(Number);
    if (v.length < 2 || v.length > 20 || v.some(function (x) { return !isFinite(x) || x % 1 !== 0; })) {
      textoEl.textContent = "Escribe entre 2 y 20 números enteros separados por comas.";
      return;
    }
    cargar(v);
  }

  raiz.querySelector(".nuevo").addEventListener("click", aleatorios);
  raiz.querySelector(".usar").addEventListener("click", propios);
  inMios.addEventListener("keydown", function (e) { if (e.key === "Enter") propios(); });
  btnAnt.addEventListener("click", function () { pausar(); if (actual > 0) mostrar(actual - 1); });
  btnSig.addEventListener("click", function () { pausar(); if (actual < pasos.length - 1) mostrar(actual + 1); });
  btnPlay.addEventListener("click", function () { if (timer) pausar(); else reproducir(); });

  aleatorios();
})();
</script>
"""


def simulador_quick_sort_numeros():
    """Muestra el simulador interactivo de Quick Sort en la celda."""
    uid = "qs" + uuid.uuid4().hex[:8]
    display(HTML(PLANTILLA.replace("__UID__", uid)))

## Ejecutar el simulador

In [2]:
simulador_quick_sort_numeros()